### Mouse brain - subregions of each brain region (mp_mouse_brain03)

For each of the 17 brain regions of mp_mouse_brain01 (**mouse_brain_regions.tsv**), the Visium spots of the region are clustered on **gene expression** (own UMAP per region) to look for **subregions**, which are then **mapped on the tissue** and described.

**Outputs** (all figures are Plotly, interactive here and saved as HTML in **root_figure**)

- **table region**: region_id, signature genes, functional description (+ size, enriched cell types)
- **table subregion**: region_id, sub_region_id, subregion signature genes, other genes, not present genes, functional description (+ diagnostics)
- long table of the 4 gene sets of every subregion (**.tsv.gz**)

**Data**: the paper's cell2location map (2 sections: ST8059048 mouse 1, ST8059051 mouse 2; 5,396 spots, 31,040 genes, raw counts) with the region labels of mp_mouse_brain01.

### Critique of the request and how it is handled

1. **Gene sets must partition the genes** (total = region signature + subregion signature + not present + other). Implemented as 4 disjoint sets per subregion; a check asserts that the counts add up.
2. **"Not present" needs a definition.** Most genes are zero in most Visium spots, so "absent" alone would list thousands of genes that are absent everywhere. Here: detected in **>= 30%** of the rest of the region, in **<= 5%** of the subregion, and significantly lower (Wilcoxon, BH FDR <= 0.05).
3. **"Other genes" is very large** (~14,000 detected genes per region). The table shows its count and the most expressed members (genes shared by the whole region). The full list is in the long gene-set file.
4. **Circularity.** The regions were made by clustering cell-type abundances; re-clustering abundances would rediscover the merged clusters. Subregions are therefore clustered on **gene expression**; abundances are used only to describe them.
5. **Two sections from two mice.** Clusters can split by section instead of anatomy: section correction (ComBat on the HVGs) and a **single-section** flag.
6. **A UMAP cluster is not a subregion until it is mapped.** Every subregion is drawn on the H&E and gets a **spatial coherence ratio** (fraction of its spots' neighbours in the same subregion / chance).
7. **Confounders found while doing it**: **sequencing depth** (low- or high-UMI spots form their own clusters and inflate gene-set sizes) and **borders** (55-µm spots at a region edge mix two regions). Both are reported (**depth_ratio**, **border_region**, flags in **status**).
8. **Resolution limits.** Small or thin regions (HPC_CA1 68 spots; cortical layers 1-2 spots thick) cannot be subdivided reliably; a minimum subregion size is enforced.
9. **edge_LowQ** is a tissue-rim artefact, not anatomy: it is in the region table but not subclustered.
10. **Functional descriptions are interpretations.** They are anchored on signature genes, enriched cell types (cell2location), the bordering region and the flags, and kept in an editable table (**mouse_brain_subregion_descriptions.tsv**); entries whose signature changed after a parameter change are marked **CHECK**.

## Loading packages

In [ ]:
import os, sys
from pathlib import Path
from importlib.metadata import version
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import plotly.io as pio
pio.renderers.default = "notebook_connected"      # interactive figures in the notebook (plotly.js from CDN)

ROOT0 = Path('/home/flavio/uv/cell_cell_communication')
ROOT_SRC = ROOT0 / 'src'
if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

from libs.single_cell_neuro_lib import SingleCellNeuro, NEURO_PARAMS, NEURO_FILES

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 250)
pd.set_option('display.max_rows', 200)
print({p: version(p) for p in ['scanpy', 'anndata', 'plotly']})

## Parameters

Parameters: **NEURO_PARAMS** / **DEFAULT_PARAMS** (subclustering: **sub_**; gene sets and flags: **region_sig_**, **sub_sig_**, **notpres_**, **subregion_**, ...; region colours: **region_colors**, light anatomical palette); override with `SingleCellNeuro(..., params=dict(...))`.

Files: **NEURO_FILES['inputs']** + **NEURO_FILES['mb03']** (`analysis='mb03'`), read and written by the class. Every method that writes a table or a figure has **force** and **verbose** (default False): with `force=False` an existing file is read (tables) or kept (figures) instead of recomputed; set `force = True` after changing parameters or inputs.

In [ ]:
force=True       # first run of this version: True (writes the new mb03_* files); then False (reads them)
verbose=True

force=False
verbose=False

scn = SingleCellNeuro(ROOT0, analysis='mb03')
print(scn)
scn.calc_df_files()

### 1. Spots, regions and expression

**prepare_expression**: gene symbols as **var_names**, raw counts in **layers['counts']**, **X** = log1p(CP10k). The paper map also carries **.raw** (Ensembl names), which is dropped so every test runs on **X**.

In [ ]:
adata_map = scn.load_paper_map('c2l_map')
df_regions = scn.calc_df_regions(adata_map)
scn.add_region_labels(df_regions, adata_map)               # params['region_label_override']: 11 -> CTX_L1, 21 -> edge_LowQ
df_abund = scn.calc_df_abundance(adata_map)

adata_expr = scn.prepare_expression(adata_map)
df_brain_regions = scn.read_tsv('region_info', verbose=verbose)
print(adata_expr)
adata_expr.obs['region_label'].value_counts().to_frame('n_spots').T

In [ ]:
# sequencing depth and cell density per region (edge_LowQ: low-quality edge)
df_depth = scn.calc_df_region_depth(adata_expr, df_abund)
df_depth.T

### 2. Region signatures (table region)

Each region vs all other spots (Wilcoxon on log1p CP10k): signature = logFC >= 1, FDR <= 0.05, detected in >= 25% of the region's spots. Enriched cell types: mean cell2location density in the region / in all other spots.

In [ ]:
force=False
verbose=False

df_rank_region, region_sig = scn.calc_df_region_signature(adata_expr, force=force, verbose=verbose)
df_region_table = scn.calc_df_region_table(adata_expr, df_rank_region, region_sig, df_abund, df_brain_regions,
                                           force=force, verbose=verbose)    # + median_UMIs, median_cells_per_spot

cols = ['region_id', 'region_name', 'n_spots', 'median_UMIs', 'median_cells_per_spot', 'n_signature_genes', 'signature_genes', 'enriched_cell_types']
print(df_region_table.shape)
df_region_table[cols]

In [ ]:
force=False
verbose=False

fig = scn.plotly_region_signature_heatmap(adata_expr, df_rank_region, n_per_region=3,
                                          fontsize_title=22, fontsize_xticks=15, fontsize_yticks=15,
                                          fontsize_colorbar=14, fontsize_hover=13, font_color="#f0f0f0", bg_color="#3a3a3a",
                                          height=1250, width=1400, force=force, verbose=verbose)
fig.show()

**Region map**: all regions of each section on the H&E (light anatomical palette). Hover: region name, spots, median UMIs, median cells per spot, signature genes, enriched cell types; click a legend entry to hide a region, double-click to isolate it.

In [ ]:
force=False
verbose=False

for lib in scn.library_ids(adata_map):
    fig = scn.plotly_region_map(adata_expr, adata_map, lib, df_region_table=df_region_table,
                                fontsize_title=22, fontsize_legend=16, fontsize_hover=13, font_color="#f0f0f0", bg_color="#3a3a3a", image_bg_color="#3a3a3a",
                                force=force, verbose=verbose)
    fig.show()

### 3. Subclustering each region (own UMAP)

**subcluster_adaptive** per region: genes detected in >= 2% of the region's spots, 2,000 HVGs (per section), ComBat by section, PCA (20), kNN (15), UMAP, Leiden over **sub_resolutions**. The resolution kept is the highest at which every subregion has >= **sub_min_size** spots and there are <= **sub_max_k** subregions.

In [ ]:
force=False
verbose=False

subs, df_grid = scn.run_subregions(adata_expr, force=force, verbose=verbose)

cols = ['region_id', 'resolution', 'k', 'min_size']
df_grid[df_grid.chosen][cols]

### 4. Gene sets of each subregion (partition)

Per subregion (Wilcoxon, subregion vs rest of its region), universe = genes detected in >= 1% of the region's spots:

| set | definition |
|---|---|
| region_signature | in the region signature (section 2) |
| subregion_signature | logFC >= 0.75, FDR <= 0.05, detected in >= 20% of the subregion; not in the region signature |
| not_present | detected in >= 30% of the rest of the region, <= 5% of the subregion, FDR <= 0.05 |
| other | the rest of the universe |

In [ ]:
force=False
verbose=False

df_gene_sets = scn.calc_df_subregion_gene_sets(adata_expr, region_sig, force=force, verbose=verbose)
df_set_counts = df_gene_sets.groupby(['sub_region_id', 'gene_set'], sort=False).size().unstack(fill_value=0)
df_set_counts['total'] = df_set_counts.sum(1)
n_universe = df_gene_sets.groupby('sub_region_id', sort=False).size()
assert (df_set_counts['total'] == n_universe.loc[df_set_counts.index]).all()
print(f"{len(df_set_counts)} subregions; the 4 sets partition every universe")
df_set_counts.head(10)

### 5. Subregion table

Diagnostics: **coherence_ratio** (spatial contiguity; 1 = random), **depth_ratio** (median UMIs of the subregion / region), **border_region** / **border_frac** (most frequent other region among its spots' neighbours), section fractions, **status** (supported / weak + flags). Functional descriptions come from **mouse_brain_subregion_descriptions.tsv** (curated); missing entries get an automatic text.

In [ ]:
force=False
verbose=False

df_descriptions = scn.read_tsv('subregion_descriptions', verbose=verbose)
df_sub_table = scn.calc_df_subregion_table(adata_expr, df_gene_sets, df_abund, df_descriptions=df_descriptions,
                                           force=force, verbose=verbose)    # params['subregion_table_first'] columns first
print(df_sub_table.status.str.split(';').str[0].value_counts().to_dict())
print(df_sub_table.description_source.value_counts().to_dict())
df_sub_table[['sub_region_id', 'subregion_name', 'n_spots', 'coherence_ratio', 'depth_ratio', 'border_region',
              'n_subregion_signature', 'n_not_present', 'status']]

In [ ]:
force=False
verbose=False

fig = scn.plotly_subregion_diagnostics(df_sub_table,
                                       fontsize_title=22, fontsize_axis=18, fontsize_ticks=18,
                                       fontsize_legend=12, fontsize_legend_title=18, fontsize_hover=13,
                                       font_color="#f0f0f0", bg_color="#062F23", grid_color="#555555", height=800, width=1800,
                                       force=force, verbose=verbose)
fig.show()

### 6. Each region: UMAP and map of its subregions

Per region: UMAP coloured by subregion, UMAP coloured by section (a good subregion contains both sections), and the subregions on the H&E of each section (cropped to the region). Click a legend entry to hide / isolate a subregion in all panels; hover shows status and signature genes.

In [ ]:
force=False
verbose=False

for region, adata_sub in subs.items():
    fig = scn.plotly_region_subregions(region, adata_sub, adata_map, df_sub_table=df_sub_table,
                                       df_region_table=df_region_table,
                                       fontsize_title=22, fontsize_subplot_title=18, fontsize_axis=18,
                                       fontsize_legend=12, fontsize_colorbar=14, fontsize_hover=13, font_color="#f0f0f0", bg_color="#3a3a3a", image_bg_color="#3a3a3a",
                                       force=force, verbose=verbose)
    fig.show()

### 7. Map of all subregions

Both sections, every subregion in a shade of its region's colour (legend grouped by region; double-click a subregion to isolate it). Hover: status, signature genes, absent genes, enriched cell types. **edge_LowQ** is grey (not subclustered).

In [ ]:
force=False
verbose=False

for lib in scn.library_ids(adata_map):
    fig = scn.plotly_subregion_map(adata_expr, adata_map, lib, df_sub_table=df_sub_table,
                                   fontsize_title=22, fontsize_legend=12, fontsize_legend_title=18, fontsize_hover=13,
                                   font_color="#f0f0f0", bg_color="#3a3a3a", image_bg_color="#3a3a3a", legend_ncols=3, 
                                   width=1800, height=800, force=force, verbose=verbose)
    fig.show()

### 8. Tables

**Table region**

In [ ]:
force=False
verbose=False

# n_subregions now that the subregions exist (mb03_region_table.tsv is rewritten when it changed)
df_region_table = scn.update_region_table_subregions(df_region_table, adata_expr, force=force, verbose=verbose)
print(df_region_table.shape)
df_region_table

**Table subregion**

In [ ]:
force=False
verbose=False

# written by calc_df_subregion_table (mb03_subregion_table.tsv)
print(df_sub_table.shape)

cols=['region_id', 'sub_region_id', 'subregion_name', 'status', 'subregion_signature_genes', 'not_present_genes',
      'enriched_cell_types', 'functional_description']
df_sub_table[cols].head(6)

### Notes for interpretation

- **supported** = spatially coherent (ratio >= 1.5) with >= 3 own signature genes; **weak** subregions are not distinct from the rest of their region.

- **low-depth / high-depth** subregions are partly technical: gene-set sizes and "not present" lists are inflated. **border with X** subregions mix two regions in the same 55-µm spots.
- **single-section** subregions may be real levels present in one section only (e.g. HYPO_4 mammillary region, STR_4 lateral hypothalamus) or section artefacts.
- Several subregions reveal **region-label errors** inherited from the region clustering (e.g. STR_4 is lateral hypothalamus, HAB_1 / HAB_2 / HAB_5 are not habenula, HPC_DG_3 / HPC_DG_4 are edge artefacts). A next step is to relabel these spots and rerun.
- Subregion IDs and gene sets depend on the parameters; after changing them, check **description_source** for **CHECK** entries and update **mouse_brain_subregion_descriptions.tsv**.